# Neural Style Transfer

> Based on Stanford CS230 — C4M4, Andrew Ng / DeepLearning.AI  
> Original paper: Gatys et al. (2015) — *A Neural Algorithm of Artistic Style*

Neural Style Transfer (NST) generates an image $G$ that merges the **content** of image $C$ with the **style** of image $S$ by optimising a combined cost function.

## Learning Objectives

1. Define content cost using deep feature activations
2. Compute style cost using the Gram matrix
3. Combine into total NST cost and understand the role of $\alpha$, $\beta$
4. Implement Gram matrix computation and a cost evaluation loop

## The NST Cost Function

$$J(G) = \alpha\, J_{\text{content}}(C, G) + \beta\, J_{\text{style}}(S, G)$$

| Term | Definition |
|------|-----------|
| $J_{\text{content}}(C,G)$ | $\frac{1}{4 n_H n_W n_C}\sum_{i,j,k}\bigl(a^{[l](C)}_{ijk} - a^{[l](G)}_{ijk}\bigr)^2$ |
| $J_{\text{style}}^{[l]}(S,G)$ | $\frac{1}{(2 n_H n_W n_C)^2}\sum_{k,k'}\bigl(G^{[l](S)}_{kk'} - G^{[l](G)}_{kk'}\bigr)^2$ |
| Gram matrix | $G^{[l]}_{kk'} = \sum_{i=1}^{n_H}\sum_{j=1}^{n_W} a^{[l]}_{ijk}\, a^{[l]}_{ijk'}$ |
| $J_{\text{style}}(S,G)$ | $\sum_l \lambda^{[l]} J_{\text{style}}^{[l]}(S,G)$ |

The Gram matrix $G^{[l]} \in \mathbb{R}^{n_C \times n_C}$ captures the **co-occurrence of features** across spatial positions — it measures style, not content.

## Pipeline Overview

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 150" width="700" height="150" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="nst" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Content image -->
  <rect x="10" y="45" width="100" height="60" rx="5" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.6"/>
  <text x="60" y="72"  text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">Content C</text>
  <text x="60" y="90"  text-anchor="middle" fill="#3a7bbf" font-size="9">(photo)</text>
  <!-- Style image -->
  <rect x="10" y="10" width="100" height="30" rx="5" fill="#c678dd" fill-opacity="0.12" stroke="#c678dd" stroke-width="1.5"/>
  <text x="60" y="28"  text-anchor="middle" fill="#8a3ab8" font-size="10" font-weight="bold">Style S (artwork)</text>
  <!-- VGG arrows and boxes -->
  <rect x="185" y="10"  width="90" height="30" rx="4" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.4"/>
  <text x="230" y="29"  text-anchor="middle" fill="#a07010" font-size="9">VGG → Gram(S)</text>
  <rect x="185" y="45"  width="90" height="60" rx="4" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.4"/>
  <text x="230" y="72"  text-anchor="middle" fill="#a07010" font-size="9" font-weight="bold">VGG → a[l](C)</text>
  <text x="230" y="88"  text-anchor="middle" fill="#a07010" font-size="9">content feats</text>
  <!-- Generated image G (initialised random) -->
  <rect x="350" y="45" width="100" height="60" rx="5" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.6"/>
  <text x="400" y="72"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Generated G</text>
  <text x="400" y="90"  text-anchor="middle" fill="#3d7a5e" font-size="9">(optimise pixels)</text>
  <!-- Cost box -->
  <rect x="510" y="40" width="140" height="70" rx="6" fill="#e05c5c" fill-opacity="0.10" stroke="#e05c5c" stroke-width="1.6"/>
  <text x="580" y="65"  text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">J(G) = αJ_cont + βJ_style</text>
  <text x="580" y="83"  text-anchor="middle" fill="#b03a3a" font-size="9">minimise via</text>
  <text x="580" y="97"  text-anchor="middle" fill="#b03a3a" font-size="9">gradient descent on G</text>
  <!-- Arrows -->
  <line x1="110" y1="75"  x2="185" y2="75"  stroke="#aaa" stroke-width="1.2" marker-end="url(#nst)"/>
  <line x1="110" y1="25"  x2="185" y2="25"  stroke="#aaa" stroke-width="1.2" marker-end="url(#nst)"/>
  <line x1="275" y1="75"  x2="350" y2="75"  stroke="#aaa" stroke-width="1.2" marker-end="url(#nst)"/>
  <line x1="450" y1="75"  x2="510" y2="75"  stroke="#aaa" stroke-width="1.2" marker-end="url(#nst)"/>
  <!-- Style cost arrow from Gram to cost box -->
  <line x1="275" y1="25"  x2="480" y2="25"  stroke="#aaa" stroke-width="1.2"/>
  <line x1="480" y1="25"  x2="480" y2="55"  stroke="#aaa" stroke-width="1.2" marker-end="url(#nst)"/>
  <text x="370" y="20" text-anchor="middle" fill="#888" font-size="8">style cost</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Gram matrix ----
def gram_matrix(A):
    """A: (n_C, n_H*n_W)  →  G: (n_C, n_C)."""
    return A @ A.T

# ---- Content cost ----
def content_cost(a_C, a_G):
    """a_C, a_G: (n_H, n_W, n_C)."""
    nH, nW, nC = a_C.shape
    return np.sum((a_C - a_G)**2) / (4 * nH * nW * nC)

# ---- Style cost for one layer ----
def style_cost_layer(a_S, a_G):
    """a_S, a_G: (n_H, n_W, n_C)."""
    nH, nW, nC = a_S.shape
    A_S = a_S.reshape(-1, nC).T   # (nC, nH*nW)
    A_G = a_G.reshape(-1, nC).T
    GS  = gram_matrix(A_S)
    GG  = gram_matrix(A_G)
    return np.sum((GS - GG)**2) / (2 * nH * nW * nC)**2

np.random.seed(1)
nH, nW, nC = 4, 4, 3
a_C = np.random.randn(nH, nW, nC)
a_G = np.random.randn(nH, nW, nC)
a_S = np.random.randn(nH, nW, nC)

j_cont  = content_cost(a_C, a_G)
j_style = style_cost_layer(a_S, a_G)
alpha, beta = 10, 40
j_total = alpha * j_cont + beta * j_style

print(f"Content cost J_cont  = {j_cont:.6f}")
print(f"Style cost   J_style = {j_style:.6f}")
print(f"Total cost   J(G)    = α·J_cont + β·J_style")
print(f"           (α={alpha}, β={beta}) = {j_total:.6f}")

# ---- Simulate NST optimisation loop ----
def run_nst_loop(a_C, a_S, alpha=10, beta=40, lr=0.05, n_iter=200, seed=7):
    np.random.seed(seed)
    G = np.random.randn(*a_C.shape) * 0.01
    history = []
    for _ in range(n_iter):
        jc  = content_cost(a_C, G)
        js  = style_cost_layer(a_S, G)
        jtot = alpha * jc + beta * js
        history.append((jc, js, jtot))
        # Analytic gradient wrt G (illustrative)
        nH, nW, nC = a_C.shape
        grad_c = (G - a_C) / (2 * nH * nW * nC)
        A_G = G.reshape(-1, nC).T
        A_S = a_S.reshape(-1, nC).T
        GG  = gram_matrix(A_G)
        GS  = gram_matrix(A_S)
        dGram = (GG - GS)                           # (nC, nC)
        dA_G  = 4 * dGram @ A_G                     # (nC, nH*nW)
        grad_s = dA_G.T.reshape(nH, nW, nC) / (2 * nH * nW * nC)**2
        G -= lr * (alpha * grad_c + beta * grad_s)
    return np.array(history)

history = run_nst_loop(a_C, a_S, n_iter=200)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Neural Style Transfer — Simulated Optimisation', fontsize=13, fontweight='bold')

iters = np.arange(len(history))
labels = ['J_content', 'J_style', 'J_total']
ylabels = ['Content cost', 'Style cost', 'Total cost J(G)']
for ax, col, label, ylabel in zip(axes, range(3), labels, ylabels):
    ax.plot(iters, history[:, col], color=P[col], lw=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{label} over iterations')
    ax.grid(True)

plt.tight_layout()
plt.show()

# ---- Gram matrix visualisation ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Gram Matrices: Style Image vs Generated Image', fontsize=12, fontweight='bold')
A_S = a_S.reshape(-1, nC).T
A_G_final = a_C.reshape(-1, nC).T   # after many iterations G → C
GS  = gram_matrix(A_S)
GG  = gram_matrix(A_G_final)
for ax, G_mat, title in zip(axes, [GS, GG], ['Gram(Style S)', 'Gram(Generated G)']):
    im = ax.imshow(G_mat, cmap='coolwarm', aspect='auto')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Channel k\'')
    ax.set_ylabel('Channel k')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()
